In [3]:
"""
MSTGNet: Multi-Scale Temporal Graph Network for Video-Level Deception Detection
Preprocessing Pipeline (v10.2 - Revision Ready / Integrity First / Dataset-Driven)

Patch vs v10.1:
- Remove misleading "with Cross-Modal Attention" wording from headers.
- Add active coordinate column names: coord_cols_active.
- Add active modality slices: active_modality_slices.
- Add raw-to-active mapping: raw_to_active_indices.
- Add explicit integrity note:
    No feature standardization is applied during preprocessing;
    fold-wise standardization is fitted only on training partitions during model training.
- Add patch_existing_processed_pickle() so existing large .pkl files can be patched
  without rerunning preprocessing from raw CSV.
- Keep both observed 1,533 coordinate columns and active 1,503 feature metadata.
"""

import os
import gc
import pickle
import warnings
import re
import shutil
from pathlib import Path
from datetime import datetime
from typing import Optional, Tuple, Dict, List, Any

import numpy as np
import pandas as pd
from tqdm import tqdm


# ============================================================
# Global
# ============================================================
warnings.simplefilter("default")

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PREPROCESSING_VERSION = "10.2_revision_ready_integrity_first"
REVISION_INTEGRITY_NOTE = (
    "No feature standardization is applied during preprocessing. "
    "Fold-wise standardization is fitted only on training partitions "
    "during model training in 02_MSTGNet_experiment.ipynb."
)


# ============================================================
# Utilities
# ============================================================
def _normalize_colname(s: str) -> str:
    """Normalize column name for robust matching logic."""
    s = str(s).strip().lower()
    s = re.sub(r"[\s\-]+", "_", s)
    s = re.sub(r"_+", "_", s)
    return s


def _strip_axis_suffix(col_norm: str) -> str:
    """Remove _x/_y/_z suffix from normalized column names."""
    if col_norm.endswith(("_x", "_y", "_z")):
        return col_norm[:-2]
    return col_norm


def _is_coord_col(col: str) -> bool:
    """Coordinate columns must end with _X/_Y/_Z or _x/_y/_z."""
    c = str(col)
    return c.endswith(("_X", "_Y", "_Z", "_x", "_y", "_z"))


def _axis_of(col: str) -> str:
    c = str(col)
    if c.endswith(("_X", "_x")):
        return "X"
    if c.endswith(("_Y", "_y")):
        return "Y"
    if c.endswith(("_Z", "_z")):
        return "Z"
    return ""


def _pretty_pct(x: float) -> str:
    return f"{x:.1f}%"


def _safe_file_size_mb(path: str) -> Optional[float]:
    try:
        return float(os.path.getsize(path) / (1024 ** 2))
    except Exception:
        return None


def _safe_file_mtime(path: str) -> Optional[str]:
    try:
        return datetime.fromtimestamp(os.path.getmtime(path)).strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return None


def backup_file_if_exists(path: str, backup_suffix: str = "_backup_v1") -> Optional[str]:
    """
    Backup a file if it exists.
    Example:
        data/processed/mstgnet_I3D.pkl
        -> data/processed/mstgnet_I3D_backup_v1.pkl
    """
    src = Path(path)
    if not src.exists():
        print(f"⚠️  Backup skipped, file not found: {src}")
        return None

    backup_path = src.with_name(f"{src.stem}{backup_suffix}{src.suffix}")

    if backup_path.exists():
        print(f"✓ Backup already exists: {backup_path}")
    else:
        shutil.copy2(src, backup_path)
        print(f"✓ Backup created: {backup_path}")

    return str(backup_path)


# ============================================================
# Validation
# ============================================================
def validate_data(
    df: pd.DataFrame,
    *,
    require_integer_frame: bool = False,
    allow_duplicate_frames: bool = True,
    deduplicate_frames: bool = False,
    dedup_keep: str = "first",
) -> pd.DataFrame:
    """
    Validate data structure and integrity.
    Returns possibly deduplicated df.
    """
    print("Validating data structure...")

    required_cols = ["Video_Name", "Frame", "Class"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise AssertionError(f"Missing required columns: {missing}")

    if df[required_cols].isnull().any().any():
        bad = df[required_cols].isnull().sum().to_dict()
        raise AssertionError(f"Missing values in required columns: {bad}")

    if not df["Class"].isin([0, 1]).all():
        bad_vals = sorted(df.loc[~df["Class"].isin([0, 1]), "Class"].unique().tolist())[:10]
        raise AssertionError(f"Invalid class labels, must be 0/1. Examples: {bad_vals}")

    if not np.issubdtype(df["Frame"].dtype, np.number):
        try:
            df["Frame"] = pd.to_numeric(df["Frame"], errors="raise")
        except Exception:
            raise AssertionError("Frame column must be numeric or convertible to numeric")

    if not np.isfinite(df["Frame"].to_numpy()).all():
        raise AssertionError("Frame contains non-finite values.")

    if require_integer_frame:
        frac = df["Frame"].to_numpy() % 1.0
        if not np.allclose(frac, 0.0):
            raise AssertionError("Frame must be integer-valued when require_integer_frame=True.")

    coord_cols = [c for c in df.columns if _is_coord_col(c)]
    if len(coord_cols) == 0:
        raise AssertionError("No coordinate columns found.")

    non_numeric = [c for c in coord_cols if not np.issubdtype(df[c].dtype, np.number)]
    if non_numeric:
        try:
            df[non_numeric] = df[non_numeric].apply(pd.to_numeric, errors="raise")
        except Exception:
            raise AssertionError(f"Non-numeric coordinate columns detected: {non_numeric[:10]}")

    if df[coord_cols].isnull().any().any():
        null_counts = int(df[coord_cols].isnull().sum().sum())
        raise AssertionError(f"Missing values detected in coordinate columns. Total NaNs: {null_counts}")

    coord_np = df[coord_cols].to_numpy()
    if not np.isfinite(coord_np).all():
        raise AssertionError("Non-finite values detected in coordinate columns.")

    label_nunique = df.groupby("Video_Name")["Class"].nunique()
    inconsistent = label_nunique[label_nunique > 1]
    if len(inconsistent) > 0:
        raise AssertionError(
            "Inconsistent labels within videos detected. "
            f"Examples: {inconsistent.head(5).to_dict()}"
        )

    dup_mask = df.duplicated(subset=["Video_Name", "Frame"], keep=False)
    dup_count = int(dup_mask.sum())

    if dup_count > 0:
        msg = (
            f"Found {dup_count} duplicate rows by (Video_Name, Frame). "
            f"allow_duplicate_frames={allow_duplicate_frames}, "
            f"deduplicate_frames={deduplicate_frames}."
        )
        if not allow_duplicate_frames and not deduplicate_frames:
            raise AssertionError(msg)

        print(f"⚠️  Warning: {msg}")

        if deduplicate_frames:
            before = len(df)
            df = df.drop_duplicates(
                subset=["Video_Name", "Frame"],
                keep=dedup_keep
            ).reset_index(drop=True)
            after = len(df)
            print(f"✓ Deduplicated frames: {before} -> {after} (keep='{dedup_keep}')")

    print("✓ Data validation completed")
    return df


# ============================================================
# Loading
# ============================================================
def load_raw_dataset(file_path: str, dataset_name: str, **validate_kwargs) -> Optional[pd.DataFrame]:
    """Load raw CSV dataset."""
    print(f"\n{'=' * 70}")
    print(f"LOADING DATASET: {dataset_name}")
    print(f"{'=' * 70}")

    try:
        df = pd.read_csv(file_path)
        print(f"Loaded: {df.shape}")
        df = validate_data(df, **validate_kwargs)
        return df
    except Exception as e:
        print(f"❌ Error loading dataset: {str(e)}")
        return None


def extract_coordinate_columns(df: pd.DataFrame) -> List[str]:
    """Extract coordinate feature columns in existing order."""
    coord_cols = [c for c in df.columns if _is_coord_col(c)]
    print(f"✓ Extracted {len(coord_cols)} coordinate features")
    return coord_cols


# ============================================================
# Feature Mapping
# ============================================================
def create_feature_index_mapping(
    coord_cols: List[str],
    *,
    iris_policy: str = "standard_only",
) -> Dict[str, Any]:
    """
    Create feature index mapping.

    iris_policy:
      - standard_only: keep MediaPipe standard iris names only
      - nonstandard_only: keep nonstandard iris names only
      - both: keep both, risk duplicate semantics
    """
    if iris_policy not in {"standard_only", "nonstandard_only", "both"}:
        raise ValueError("iris_policy must be one of: standard_only, nonstandard_only, both")

    print("\nCreating feature index mapping...")
    print(f"Iris policy: {iris_policy}")

    STANDARD_IRIS_BASE_NAMES = {
        "left_iris_center",
        "left_iris_right",
        "left_iris_top",
        "left_iris_left",
        "left_iris_bottom",
        "right_iris_center",
        "right_iris_right",
        "right_iris_top",
        "right_iris_left",
        "right_iris_bottom",
    }

    face_indices: List[int] = []
    iris_standard_indices: List[int] = []
    iris_nonstandard_indices: List[int] = []
    body_indices: List[int] = []

    iris_standard_columns: List[str] = []
    iris_nonstandard_columns: List[str] = []

    axis_counts = {"X": 0, "Y": 0, "Z": 0}
    for col in coord_cols:
        ax = _axis_of(col)
        if ax:
            axis_counts[ax] += 1

    for idx, col in enumerate(coord_cols):
        col_norm = _normalize_colname(col)
        base = _strip_axis_suffix(col_norm)

        if ("pose_" in col_norm) or col_norm.startswith("pose_") or col_norm.startswith("pose"):
            body_indices.append(idx)
            continue

        if "iris" in col_norm:
            if base in STANDARD_IRIS_BASE_NAMES:
                iris_standard_indices.append(idx)
                iris_standard_columns.append(col)
            else:
                iris_nonstandard_indices.append(idx)
                iris_nonstandard_columns.append(col)
            continue

        face_indices.append(idx)

    spec = {
        "expected_face": 468 * 3,
        "expected_iris": 10 * 3,
        "expected_pose": 33 * 3,
        "expected_total": 511 * 3,
    }

    observed = {
        "face": len(face_indices),
        "iris_standard": len(iris_standard_indices),
        "iris_nonstandard": len(iris_nonstandard_indices),
        "pose": len(body_indices),
        "total": len(coord_cols),
        "axes": axis_counts,
    }

    partition_sum_raw = (
        observed["face"]
        + observed["iris_standard"]
        + observed["iris_nonstandard"]
        + observed["pose"]
    )
    partition_match_raw = partition_sum_raw == observed["total"]
    has_both_iris = observed["iris_standard"] > 0 and observed["iris_nonstandard"] > 0

    if iris_policy == "standard_only":
        active_iris_standard = iris_standard_indices
        active_iris_nonstandard = []
        dropped_iris_standard = []
        dropped_iris_nonstandard = iris_nonstandard_indices
    elif iris_policy == "nonstandard_only":
        active_iris_standard = []
        active_iris_nonstandard = iris_nonstandard_indices
        dropped_iris_standard = iris_standard_indices
        dropped_iris_nonstandard = []
    else:
        active_iris_standard = iris_standard_indices
        active_iris_nonstandard = iris_nonstandard_indices
        dropped_iris_standard = []
        dropped_iris_nonstandard = []

    active_feature_indices_flat = (
        face_indices
        + active_iris_standard
        + active_iris_nonstandard
        + body_indices
    )

    active_total = len(active_feature_indices_flat)

    n_face_active = len(face_indices)
    n_eyes_active = len(active_iris_standard) + len(active_iris_nonstandard)
    n_body_active = len(body_indices)

    active_modality_slices = {
        "face": [0, n_face_active],
        "eyes": [n_face_active, n_face_active + n_eyes_active],
        "body": [n_face_active + n_eyes_active, n_face_active + n_eyes_active + n_body_active],
    }

    verification = {
        "partition_sum_raw": int(partition_sum_raw),
        "partition_match_total_raw": bool(partition_match_raw),

        "expected_face_spec": spec["expected_face"],
        "expected_iris_spec": spec["expected_iris"],
        "expected_pose_spec": spec["expected_pose"],
        "expected_total_spec": spec["expected_total"],

        "face_matches_spec": bool(observed["face"] == spec["expected_face"]),
        "iris_standard_matches_spec": bool(observed["iris_standard"] == spec["expected_iris"]),
        "pose_matches_spec": bool(observed["pose"] == spec["expected_pose"]),
        "total_matches_spec": bool(observed["total"] == spec["expected_total"]),

        "has_nonstandard_iris": bool(observed["iris_nonstandard"] > 0),
        "has_both_iris_namings": bool(has_both_iris),
        "axis_counts": axis_counts,

        "active_total": int(active_total),
        "active_modality_slices": active_modality_slices,
    }

    print(f"\n{'=' * 70}")
    print("FEATURE MAPPING REPORT (v10.2 - revision ready)")
    print(f"{'=' * 70}")
    print("Observed feature counts:")
    print(f"  Face:            {observed['face']}")
    print(f"  Iris standard:   {observed['iris_standard']}")
    print(f"  Iris non-std:    {observed['iris_nonstandard']}")
    print(f"  Pose/body:       {observed['pose']}")
    print(f"  TOTAL observed:  {observed['total']}")
    print(f"  Axes:            X={axis_counts['X']}, Y={axis_counts['Y']}, Z={axis_counts['Z']}")

    print("\nReference MediaPipe spec:")
    print(f"  Face expected:   {spec['expected_face']}")
    print(f"  Iris expected:   {spec['expected_iris']}")
    print(f"  Pose expected:   {spec['expected_pose']}")
    print(f"  Total expected:  {spec['expected_total']}")

    if observed["iris_nonstandard"] > 0:
        print(f"\n⚠️  Found {observed['iris_nonstandard']} non-standard iris columns.")
        for c in iris_nonstandard_columns[:10]:
            print(f"     ⚠️  {c}")
        if len(iris_nonstandard_columns) > 10:
            print(f"     ... and {len(iris_nonstandard_columns) - 10} more")

    if has_both_iris:
        print("\n⚠️  Iris appears in BOTH standard and non-standard naming.")
        print("    Current policy keeps only selected active iris columns.")
        print(f"    iris_policy = {iris_policy}")

    print(f"\n{'=' * 70}")
    print("VERIFICATION STATUS")
    print(f"{'=' * 70}")
    print(f"  {'✅' if partition_match_raw else '❌'} Raw partition sum: {partition_sum_raw}/{observed['total']}")
    print(f"  {'✅' if observed['iris_standard'] == spec['expected_iris'] else '⚠️ '} Iris standard vs spec: {observed['iris_standard']}/{spec['expected_iris']}")
    print(f"  {'✅' if observed['pose'] == spec['expected_pose'] else '⚠️ '} Pose vs spec: {observed['pose']}/{spec['expected_pose']}")
    print(f"  {'✅' if observed['face'] == spec['expected_face'] else '⚠️ '} Face vs spec: {observed['face']}/{spec['expected_face']}")
    print(f"  {'✅' if observed['total'] == spec['expected_total'] else '⚠️ '} Total vs spec: {observed['total']}/{spec['expected_total']}")
    print(f"  Active total: {active_total}/{observed['total']}")
    print(f"  Active slices: {active_modality_slices}")
    print(f"{'=' * 70}\n")

    feature_mapping = {
        # Raw partitions
        "face_indices": face_indices,
        "body_indices": body_indices,
        "iris_standard_indices": iris_standard_indices,
        "iris_nonstandard_indices": iris_nonstandard_indices,

        # Active partitions
        "active_face_indices": face_indices,
        "active_body_indices": body_indices,
        "active_iris_standard_indices": active_iris_standard,
        "active_iris_nonstandard_indices": active_iris_nonstandard,

        # Active convenience
        "active_feature_indices_flat": active_feature_indices_flat,
        "raw_to_active_indices": active_feature_indices_flat,
        "active_modality_slices": active_modality_slices,

        "counts": {
            "observed": {
                "face": len(face_indices),
                "pose": len(body_indices),
                "iris_standard": len(iris_standard_indices),
                "iris_nonstandard": len(iris_nonstandard_indices),
                "total": len(coord_cols),
            },
            "active": {
                "face": len(face_indices),
                "pose": len(body_indices),
                "iris_standard": len(active_iris_standard),
                "iris_nonstandard": len(active_iris_nonstandard),
                "eyes": n_eyes_active,
                "active_total": int(active_total),
            },
            "dropped_by_policy": {
                "dropped_iris_standard": len(dropped_iris_standard),
                "dropped_iris_nonstandard": len(dropped_iris_nonstandard),
            },
        },

        "iris_policy": iris_policy,

        # Audit
        "iris_standard_columns": iris_standard_columns,
        "iris_nonstandard_columns": iris_nonstandard_columns,

        "verification": verification,

        "notes": [
            "Use active_feature_indices_flat for all downstream experiments.",
            "Active features exclude duplicated or non-selected iris columns according to iris_policy.",
            "Spec fields are informational only; dataset-driven counts are authoritative.",
            REVISION_INTEGRITY_NOTE,
        ],
    }

    return feature_mapping


def derive_active_feature_metadata(
    coord_cols_observed: List[str],
    feature_mapping: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Derive revision-ready active feature metadata from observed coordinate columns
    and feature_mapping.

    Returns:
      coord_cols_active
      active_modality_slices
      raw_to_active_indices
      active_feature_count
    """
    active_idx = feature_mapping.get("active_feature_indices_flat")
    if active_idx is None:
        raise KeyError("feature_mapping['active_feature_indices_flat'] not found.")

    active_idx = list(active_idx)

    if len(active_idx) == 0:
        raise ValueError("active_feature_indices_flat is empty.")

    if max(active_idx) >= len(coord_cols_observed):
        raise ValueError(
            "active_feature_indices_flat contains index outside coord_cols_observed range."
        )

    coord_cols_active = [coord_cols_observed[i] for i in active_idx]

    n_face = len(feature_mapping.get("active_face_indices", []))
    n_eyes = (
        len(feature_mapping.get("active_iris_standard_indices", []))
        + len(feature_mapping.get("active_iris_nonstandard_indices", []))
    )
    n_body = len(feature_mapping.get("active_body_indices", []))

    active_modality_slices = {
        "face": [0, n_face],
        "eyes": [n_face, n_face + n_eyes],
        "body": [n_face + n_eyes, n_face + n_eyes + n_body],
    }

    active_total_from_slices = active_modality_slices["body"][1]
    if active_total_from_slices != len(active_idx):
        raise AssertionError(
            "Active modality slices do not match active feature count: "
            f"slices_total={active_total_from_slices}, active_idx={len(active_idx)}"
        )

    if verbose:
        print("✓ Derived active feature metadata")
        print(f"  coord_cols_observed: {len(coord_cols_observed)}")
        print(f"  coord_cols_active:   {len(coord_cols_active)}")
        print(f"  active slices:       {active_modality_slices}")

    return {
        "coord_cols_active": coord_cols_active,
        "active_modality_slices": active_modality_slices,
        "raw_to_active_indices": active_idx,
        "active_feature_count": int(len(active_idx)),
    }


# ============================================================
# Organize by Video
# ============================================================
def organize_by_video(df: pd.DataFrame, coord_cols: List[str]) -> Dict[str, Dict[str, Any]]:
    """Organize frame-level data into video-level structure."""
    print("\nOrganizing data by video...")

    video_to_frames: Dict[str, Dict[str, Any]] = {}
    grouped = df.groupby("Video_Name", sort=False)

    for video_name, video_data in tqdm(grouped, total=grouped.ngroups, desc="Processing videos"):
        video_data = video_data.sort_values("Frame").reset_index(drop=True)

        features = video_data[coord_cols].to_numpy(dtype=np.float32, copy=True)
        label = int(video_data["Class"].iloc[0])
        n_frames = int(len(video_data))

        video_to_frames[str(video_name)] = {
            "features": features,
            "label": label,
            "n_frames": n_frames,
        }

    print(f"✓ Organized {len(video_to_frames)} videos")
    return video_to_frames


# ============================================================
# Statistics
# ============================================================
def calculate_dataset_statistics(video_to_frames: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    """Calculate dataset statistics."""
    print("\nCalculating dataset statistics...")

    total_videos = int(len(video_to_frames))
    total_frames = int(sum(v["n_frames"] for v in video_to_frames.values()))

    labels_video = [v["label"] for v in video_to_frames.values()]
    class_0_v = int(sum(1 for l in labels_video if l == 0))
    class_1_v = int(sum(1 for l in labels_video if l == 1))

    class_0_f = int(sum(v["n_frames"] for v in video_to_frames.values() if v["label"] == 0))
    class_1_f = int(sum(v["n_frames"] for v in video_to_frames.values() if v["label"] == 1))

    frame_counts = [v["n_frames"] for v in video_to_frames.values()]

    stats = {
        "total_videos": total_videos,
        "total_frames": total_frames,
        "class_distribution_per_video": {
            "class_0": class_0_v,
            "class_1": class_1_v,
            "class_0_pct": float((class_0_v / total_videos) * 100) if total_videos else 0.0,
            "class_1_pct": float((class_1_v / total_videos) * 100) if total_videos else 0.0,
        },
        "class_distribution_per_frame": {
            "class_0_frames": class_0_f,
            "class_1_frames": class_1_f,
            "class_0_pct": float((class_0_f / total_frames) * 100) if total_frames else 0.0,
            "class_1_pct": float((class_1_f / total_frames) * 100) if total_frames else 0.0,
        },
        "frames_per_video": {
            "mean": float(np.mean(frame_counts)) if frame_counts else 0.0,
            "median": float(np.median(frame_counts)) if frame_counts else 0.0,
            "min": int(np.min(frame_counts)) if frame_counts else 0,
            "max": int(np.max(frame_counts)) if frame_counts else 0,
            "std": float(np.std(frame_counts, ddof=1)) if len(frame_counts) > 1 else 0.0,
            "q25": float(np.percentile(frame_counts, 25)) if frame_counts else 0.0,
            "q75": float(np.percentile(frame_counts, 75)) if frame_counts else 0.0,
        },
    }

    print(f"\n{'=' * 50}")
    print("DATASET STATISTICS")
    print(f"{'=' * 50}")
    print(f"Total videos: {stats['total_videos']}")
    print("Class distribution per video:")
    print(f"  Class 0: {stats['class_distribution_per_video']['class_0']} "
          f"({_pretty_pct(stats['class_distribution_per_video']['class_0_pct'])})")
    print(f"  Class 1: {stats['class_distribution_per_video']['class_1']} "
          f"({_pretty_pct(stats['class_distribution_per_video']['class_1_pct'])})")
    print(f"Total frames: {stats['total_frames']}")
    print("Frames/video:")
    print(f"  Mean:   {stats['frames_per_video']['mean']:.2f}")
    print(f"  Median: {stats['frames_per_video']['median']:.2f}")
    print(f"  Min:    {stats['frames_per_video']['min']}")
    print(f"  Max:    {stats['frames_per_video']['max']}")
    print(f"  Std:    {stats['frames_per_video']['std']:.2f}")
    print(f"{'=' * 50}\n")

    return stats


def create_video_arrays(
    video_to_frames: Dict[str, Dict[str, Any]],
    *,
    sort_video_ids: bool = False
) -> Tuple[np.ndarray, np.ndarray]:
    """Create numpy arrays for video IDs and labels."""
    video_ids = list(video_to_frames.keys())
    if sort_video_ids:
        video_ids = sorted(video_ids)

    video_ids_arr = np.array(video_ids)
    video_labels = np.array([video_to_frames[vid]["label"] for vid in video_ids_arr], dtype=np.int8)

    print(f"✓ Created video arrays: {len(video_ids_arr)} videos (sorted={sort_video_ids})")
    return video_ids_arr, video_labels


# ============================================================
# Experiment Config
# ============================================================
def create_experiment_config() -> Dict[str, Any]:
    """Create experiment configuration for reproducibility."""
    return {
        "random_seed": 42,
        "preprocessing_params": {
            "data_type": "float32",
            "sort_by_frame": True,
            "frame_rate_assumption": 30,
            "integrity_policy": "dataset_driven_counts + finite_checks + iris_policy",
            "standardization": "None in preprocessing; fold-wise train-only scaling during training.",
        },
        "recommended_training_params": {
            "sequence_length_final": 50,
            "stride_final": 25,
            "batch_size_final": 16,
            "n_folds": 5,
            "threshold_strategy": "Youden's J from validation fold",
            "early_stopping_metric": "validation AUC",
            "normalization": "StandardScaler fit on train fold only, transform val/test",
            "learning_rate": 1e-4,
            "weight_decay": 1e-2,
            "dropout_i3d": 0.4,
            "dropout_rlt": 0.3,
        },
        "model_architecture_revision": {
            "full_model_use_cross_modal": False,
            "cross_modal_attention_role": "ablation_variant_only",
            "hidden_features": 128,
            "num_heads": 4,
            "graph_type": "landmark-derived temporal graph",
            "multi_scale_branches": [
                {"name": "short_local", "kernel": 3, "dilation": 1, "effective_rf_frames": 3},
                {"name": "medium_local", "kernel": 5, "dilation": 1, "effective_rf_frames": 5},
                {"name": "dilated_longer_local", "kernel": 3, "dilation": 2, "effective_rf_frames": 5},
                {"name": "global_pointwise", "kernel": 1, "dilation": 1, "effective_rf_frames": 1},
            ],
        },
    }


# ============================================================
# Save Summary
# ============================================================
def write_summary_file(
    *,
    summary_path: str,
    dataset_name: str,
    metadata: Dict[str, Any],
    stats: Dict[str, Any],
    coord_cols_observed: List[str],
    coord_cols_active: List[str],
    feature_mapping: Dict[str, Any],
    file_size_mb: float,
) -> None:
    """Write human-readable summary text file."""
    fm = feature_mapping
    obs = fm["counts"]["observed"]
    act = fm["counts"]["active"]
    ver = fm["verification"]

    active_modality_slices = fm.get("active_modality_slices", metadata["feature_info"].get("active_modality_slices"))

    with open(summary_path, "w", encoding="utf-8") as f:
        f.write(f"{'=' * 70}\n")
        f.write(f"MSTGNet DATA SUMMARY: {dataset_name}\n")
        f.write(f"{'=' * 70}\n")
        f.write(f"Created: {metadata['creation_date']}\n")
        f.write(f"Model: {metadata['model']}\n")
        f.write(f"Version: {metadata['preprocessing_version']}\n\n")

        f.write("INTEGRITY NOTES:\n")
        for note in metadata["integrity_notes"]:
            f.write(f"  - {note}\n")
        f.write("\n")

        f.write("DATASET OVERVIEW:\n")
        f.write(f"  Total videos: {stats['total_videos']}\n")
        f.write(f"  Total frames: {stats['total_frames']}\n")
        f.write(f"  Observed features/frame: {len(coord_cols_observed)}\n")
        f.write(f"  Active features/frame:   {len(coord_cols_active)}\n\n")

        f.write("FEATURE BREAKDOWN OBSERVED:\n")
        f.write(f"  Face: {obs['face']}\n")
        f.write(f"  Iris standard: {obs['iris_standard']}\n")
        f.write(f"  Iris non-standard: {obs['iris_nonstandard']}\n")
        f.write(f"  Pose/body: {obs['pose']}\n")
        f.write(f"  Total observed: {obs['total']}\n\n")

        f.write("FEATURE BREAKDOWN ACTIVE USED DOWNSTREAM:\n")
        f.write(f"  iris_policy: {fm['iris_policy']}\n")
        f.write(f"  Face: {act['face']}\n")
        f.write(f"  Eyes/iris standard: {act['iris_standard']}\n")
        f.write(f"  Eyes/iris non-standard: {act['iris_nonstandard']}\n")
        f.write(f"  Pose/body: {act['pose']}\n")
        f.write(f"  Active total: {act['active_total']}\n")
        f.write(f"  Active modality slices: {active_modality_slices}\n")
        f.write(f"  Dropped by policy: {fm['counts']['dropped_by_policy']}\n\n")

        f.write("SPEC REFERENCE NOT ENFORCED:\n")
        f.write(f"  Face expected: {ver['expected_face_spec']}\n")
        f.write(f"  Iris expected: {ver['expected_iris_spec']}\n")
        f.write(f"  Pose expected: {ver['expected_pose_spec']}\n")
        f.write(f"  Total expected: {ver['expected_total_spec']}\n\n")

        f.write("VERIFICATION:\n")
        f.write(f"  Raw partition match total: {ver['partition_match_total_raw']} "
                f"({ver['partition_sum_raw']}/{obs['total']})\n")
        f.write(f"  Iris standard matches spec: {ver['iris_standard_matches_spec']}\n")
        f.write(f"  Pose matches spec: {ver['pose_matches_spec']}\n")
        f.write(f"  Face matches spec: {ver['face_matches_spec']}\n")
        f.write(f"  Total matches spec: {ver['total_matches_spec']}\n")
        f.write(f"  Has non-standard iris: {ver['has_nonstandard_iris']}\n")
        f.write(f"  Has both iris namings: {ver['has_both_iris_namings']}\n")
        f.write(f"  Axis counts: {ver['axis_counts']}\n\n")

        f.write("CLASS DISTRIBUTION PER VIDEO:\n")
        f.write(f"  Class 0: {stats['class_distribution_per_video']['class_0']} "
                f"({_pretty_pct(stats['class_distribution_per_video']['class_0_pct'])})\n")
        f.write(f"  Class 1: {stats['class_distribution_per_video']['class_1']} "
                f"({_pretty_pct(stats['class_distribution_per_video']['class_1_pct'])})\n\n")

        f.write("CLASS DISTRIBUTION PER FRAME:\n")
        f.write(f"  Class 0 frames: {stats['class_distribution_per_frame']['class_0_frames']} "
                f"({_pretty_pct(stats['class_distribution_per_frame']['class_0_pct'])})\n")
        f.write(f"  Class 1 frames: {stats['class_distribution_per_frame']['class_1_frames']} "
                f"({_pretty_pct(stats['class_distribution_per_frame']['class_1_pct'])})\n\n")

        f.write("FRAMES PER VIDEO:\n")
        f.write(f"  Mean: {stats['frames_per_video']['mean']:.2f}\n")
        f.write(f"  Median: {stats['frames_per_video']['median']:.2f}\n")
        f.write(f"  Min: {stats['frames_per_video']['min']}\n")
        f.write(f"  Max: {stats['frames_per_video']['max']}\n")
        f.write(f"  Std: {stats['frames_per_video']['std']:.2f}\n\n")

        f.write(f"FILE SIZE: {file_size_mb:.2f} MB\n")
        f.write(f"{'=' * 70}\n")


# ============================================================
# Save Data
# ============================================================
def save_mstgnet_data(
    video_to_frames: Dict[str, Dict[str, Any]],
    video_ids: np.ndarray,
    video_labels: np.ndarray,
    coord_cols: List[str],
    feature_mapping: Dict[str, Any],
    stats: Dict[str, Any],
    dataset_name: str,
    *,
    source_csv_path: Optional[str] = None,
    source_csv_shape: Optional[Tuple[int, int]] = None,
) -> None:
    """Save preprocessed data for MSTGNet training."""
    print(f"\n{'=' * 70}")
    print(f"SAVING MSTGNet DATA: {dataset_name}")
    print(f"{'=' * 70}")

    experiment_config = create_experiment_config()

    active_meta = derive_active_feature_metadata(coord_cols, feature_mapping, verbose=True)

    coord_cols_active = active_meta["coord_cols_active"]
    active_modality_slices = active_meta["active_modality_slices"]
    raw_to_active_indices = active_meta["raw_to_active_indices"]

    # Store in feature_mapping too for downstream convenience
    feature_mapping["active_modality_slices"] = active_modality_slices
    feature_mapping["raw_to_active_indices"] = raw_to_active_indices
    feature_mapping["active_feature_indices_flat"] = raw_to_active_indices
    feature_mapping["verification"]["active_modality_slices"] = active_modality_slices
    feature_mapping["verification"]["active_total"] = int(len(raw_to_active_indices))

    metadata = {
        "creation_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "dataset_name": dataset_name,
        "model": "MSTGNet",
        "preprocessing_version": PREPROCESSING_VERSION,
        "description": "Multi-Scale Temporal Graph Network for video-level deception detection",
        "source_info": {
            "source_csv_path": source_csv_path,
            "source_csv_shape": tuple(source_csv_shape) if source_csv_shape is not None else None,
            "source_csv_size_mb": _safe_file_size_mb(source_csv_path) if source_csv_path else None,
            "source_csv_mtime": _safe_file_mtime(source_csv_path) if source_csv_path else None,
        },
        "integrity_notes": [
            "Counts are derived from the CSV dataset. MediaPipe spec is reported as reference only.",
            "Finite checks enforced for Frame and coordinate values; no NaN/inf allowed.",
            "Iris naming duplication risk handled via iris_policy; use active_feature_indices_flat downstream.",
            "Observed coordinate features are retained for audit; active features are used for model training.",
            "No temporal resampling is performed; frame_rate_assumption is informational.",
            REVISION_INTEGRITY_NOTE,
        ],
        "feature_info": {
            "n_features_observed": int(len(coord_cols)),
            "n_features_active": int(len(coord_cols_active)),
            "feature_names_observed": coord_cols,
            "feature_names_active": coord_cols_active,
            "coord_cols_active": coord_cols_active,
            "active_modality_slices": active_modality_slices,
            "raw_to_active_indices": raw_to_active_indices,
            "data_type": "float32",
            "feature_breakdown": feature_mapping,
        },
        "statistics": stats,
        "experiment_config": experiment_config,
    }

    mstgnet_data = {
        "video_to_frames": video_to_frames,
        "video_ids": video_ids,
        "video_labels": video_labels,

        # Observed/raw coordinate columns
        "coord_cols_observed": coord_cols,

        # Revision-ready active metadata
        "coord_cols_active": coord_cols_active,
        "active_modality_slices": active_modality_slices,
        "raw_to_active_indices": raw_to_active_indices,

        "feature_mapping": feature_mapping,
        "metadata": metadata,
    }

    output_path = PROCESSED_DIR / f"mstgnet_{dataset_name}.pkl"
    with open(output_path, "wb") as f:
        pickle.dump(mstgnet_data, f, protocol=pickle.HIGHEST_PROTOCOL)

    file_size_mb = os.path.getsize(output_path) / (1024 ** 2)

    print(f"✓ Saved: {output_path}")
    print(f"  File size: {file_size_mb:.2f} MB")

    summary_path = PROCESSED_DIR / f"mstgnet_{dataset_name}_summary.txt"
    write_summary_file(
        summary_path=str(summary_path),
        dataset_name=dataset_name,
        metadata=metadata,
        stats=stats,
        coord_cols_observed=coord_cols,
        coord_cols_active=coord_cols_active,
        feature_mapping=feature_mapping,
        file_size_mb=file_size_mb,
    )

    print(f"✓ Saved: {summary_path}")
    print(f"{'=' * 70}\n")


# ============================================================
# Full Preprocessing Pipeline
# ============================================================
def preprocess_for_mstgnet(
    file_path: str,
    dataset_name: str,
    *,
    iris_policy: str = "standard_only",
    require_integer_frame: bool = False,
    allow_duplicate_frames: bool = True,
    deduplicate_frames: bool = False,
    dedup_keep: str = "first",
    sort_video_ids: bool = False,
) -> Optional[Dict[str, Any]]:
    print(f"\n{'=' * 70}")
    print(f"PREPROCESSING FOR MSTGNet: {dataset_name}")
    print(f"{'=' * 70}")
    print("Multi-Scale Temporal Graph Network")
    print(f"Version: {PREPROCESSING_VERSION}")
    print(f"{'=' * 70}\n")

    df = load_raw_dataset(
        file_path,
        dataset_name,
        require_integer_frame=require_integer_frame,
        allow_duplicate_frames=allow_duplicate_frames,
        deduplicate_frames=deduplicate_frames,
        dedup_keep=dedup_keep,
    )
    if df is None:
        return None

    source_csv_shape = df.shape

    coord_cols = extract_coordinate_columns(df)
    feature_mapping = create_feature_index_mapping(coord_cols, iris_policy=iris_policy)
    video_to_frames = organize_by_video(df, coord_cols)

    del df
    gc.collect()

    stats = calculate_dataset_statistics(video_to_frames)
    video_ids, video_labels = create_video_arrays(video_to_frames, sort_video_ids=sort_video_ids)

    save_mstgnet_data(
        video_to_frames=video_to_frames,
        video_ids=video_ids,
        video_labels=video_labels,
        coord_cols=coord_cols,
        feature_mapping=feature_mapping,
        stats=stats,
        dataset_name=dataset_name,
        source_csv_path=file_path,
        source_csv_shape=source_csv_shape,
    )

    active_meta = derive_active_feature_metadata(coord_cols, feature_mapping, verbose=False)

    print(f"{'=' * 70}")
    print(f"PREPROCESSING COMPLETED: {dataset_name}")
    print(f"{'=' * 70}")
    print(f"✅ Videos organized: {len(video_ids)}")
    print(f"✅ Observed coordinate features: {len(coord_cols)}")
    print(f"✅ Active coordinate features:   {active_meta['active_feature_count']}")
    print(f"✅ Active modality slices:       {active_meta['active_modality_slices']}")

    ver = feature_mapping["verification"]
    if not ver["partition_match_total_raw"]:
        print("❌ Raw partition mismatch: indices do not sum to observed total features.")
    else:
        print("✅ Raw partition integrity: PASSED")

    if ver["has_both_iris_namings"] and iris_policy == "both":
        print("⚠️  Both iris naming schemes are retained. This may duplicate semantics.")

    print("✅ Ready for MSTGNet training with revision-ready metadata")
    print(f"{'=' * 70}\n")

    return {
        "video_to_frames": video_to_frames,
        "video_ids": video_ids,
        "video_labels": video_labels,
        "coord_cols_observed": coord_cols,
        "coord_cols_active": active_meta["coord_cols_active"],
        "active_modality_slices": active_meta["active_modality_slices"],
        "raw_to_active_indices": active_meta["raw_to_active_indices"],
        "feature_mapping": feature_mapping,
        "stats": stats,
    }


# ============================================================
# Patch Existing Pickle Without Rerun
# ============================================================
def patch_existing_processed_pickle(
    dataset_name: str,
    *,
    processed_dir: str = "data/processed",
    backup: bool = True,
    backup_suffix: str = "_backup_v1",
    expected_active_total: Optional[int] = 1503,
) -> Dict[str, Any]:
    """
    Patch existing mstgnet_{dataset}.pkl with v10.2 revision-ready metadata.

    This avoids rerunning preprocessing from raw CSV.

    Adds top-level keys:
      - coord_cols_active
      - active_modality_slices
      - raw_to_active_indices

    Also updates:
      - feature_mapping
      - metadata.feature_info
      - metadata.integrity_notes
      - metadata.preprocessing_version
    """
    pkl_path = Path(processed_dir) / f"mstgnet_{dataset_name}.pkl"

    if not pkl_path.exists():
        raise FileNotFoundError(f"Processed pickle not found: {pkl_path}")

    if backup:
        backup_file_if_exists(str(pkl_path), backup_suffix=backup_suffix)

    print(f"\n{'=' * 70}")
    print(f"PATCHING EXISTING PROCESSED PICKLE: {dataset_name}")
    print(f"{'=' * 70}")
    print(f"Loading: {pkl_path}")

    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    if "coord_cols_observed" not in data:
        raise KeyError("Missing top-level key: coord_cols_observed")
    if "feature_mapping" not in data:
        raise KeyError("Missing top-level key: feature_mapping")

    coord_cols_observed = data["coord_cols_observed"]
    feature_mapping = data["feature_mapping"]

    active_meta = derive_active_feature_metadata(
        coord_cols_observed,
        feature_mapping,
        verbose=True,
    )

    coord_cols_active = active_meta["coord_cols_active"]
    active_modality_slices = active_meta["active_modality_slices"]
    raw_to_active_indices = active_meta["raw_to_active_indices"]
    active_total = active_meta["active_feature_count"]

    if expected_active_total is not None and active_total != expected_active_total:
        print(
            f"⚠️  Active total is {active_total}, expected {expected_active_total}. "
            "Continuing because dataset-driven metadata is authoritative."
        )

    # Patch top-level keys
    data["coord_cols_active"] = coord_cols_active
    data["active_modality_slices"] = active_modality_slices
    data["raw_to_active_indices"] = raw_to_active_indices

    # Patch feature_mapping
    feature_mapping["active_modality_slices"] = active_modality_slices
    feature_mapping["raw_to_active_indices"] = raw_to_active_indices
    feature_mapping["active_feature_indices_flat"] = raw_to_active_indices

    if "verification" not in feature_mapping:
        feature_mapping["verification"] = {}
    feature_mapping["verification"]["active_modality_slices"] = active_modality_slices
    feature_mapping["verification"]["active_total"] = int(active_total)

    if "notes" not in feature_mapping:
        feature_mapping["notes"] = []
    if REVISION_INTEGRITY_NOTE not in feature_mapping["notes"]:
        feature_mapping["notes"].append(REVISION_INTEGRITY_NOTE)

    data["feature_mapping"] = feature_mapping

    # Patch metadata
    metadata = data.get("metadata", {})
    metadata["preprocessing_version"] = PREPROCESSING_VERSION
    metadata["patched_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    metadata["patch_note"] = (
        "Existing processed pickle patched with v10.2 revision-ready metadata; "
        "no raw preprocessing rerun."
    )

    if "integrity_notes" not in metadata:
        metadata["integrity_notes"] = []
    if REVISION_INTEGRITY_NOTE not in metadata["integrity_notes"]:
        metadata["integrity_notes"].append(REVISION_INTEGRITY_NOTE)

    if "feature_info" not in metadata:
        metadata["feature_info"] = {}

    metadata["feature_info"]["n_features_observed"] = int(len(coord_cols_observed))
    metadata["feature_info"]["n_features_active"] = int(active_total)
    metadata["feature_info"]["feature_names_observed"] = coord_cols_observed
    metadata["feature_info"]["feature_names_active"] = coord_cols_active
    metadata["feature_info"]["coord_cols_active"] = coord_cols_active
    metadata["feature_info"]["active_modality_slices"] = active_modality_slices
    metadata["feature_info"]["raw_to_active_indices"] = raw_to_active_indices
    metadata["feature_info"]["feature_breakdown"] = feature_mapping
    metadata["feature_info"]["data_type"] = metadata["feature_info"].get("data_type", "float32")

    data["metadata"] = metadata

    # Save patched pickle
    with open(pkl_path, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

    file_size_mb = os.path.getsize(pkl_path) / (1024 ** 2)

    print(f"✓ Patched and saved: {pkl_path}")
    print(f"  File size: {file_size_mb:.2f} MB")
    print("Patch summary:")
    print(f"  Observed features: {len(coord_cols_observed)}")
    print(f"  Active features:   {active_total}")
    print(f"  Active slices:     {active_modality_slices}")
    print(f"{'=' * 70}\n")

    return {
        "dataset_name": dataset_name,
        "pkl_path": str(pkl_path),
        "observed_features": int(len(coord_cols_observed)),
        "active_features": int(active_total),
        "active_modality_slices": active_modality_slices,
        "patched": True,
    }


# ============================================================
# Sanity Check Existing Processed Pickle
# ============================================================
def sanity_check_processed_pickle(
    dataset_name: str,
    *,
    processed_dir: str = "data/processed",
    expected_observed_total: Optional[int] = 1533,
    expected_active_total: Optional[int] = 1503,
) -> Dict[str, Any]:
    """Sanity check revision-ready processed pickle."""
    pkl_path = Path(processed_dir) / f"mstgnet_{dataset_name}.pkl"

    if not pkl_path.exists():
        raise FileNotFoundError(f"Processed pickle not found: {pkl_path}")

    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    required_keys = [
        "video_to_frames",
        "video_ids",
        "video_labels",
        "coord_cols_observed",
        "coord_cols_active",
        "active_modality_slices",
        "raw_to_active_indices",
        "feature_mapping",
        "metadata",
    ]

    missing = [k for k in required_keys if k not in data]
    if missing:
        raise AssertionError(f"Missing required keys in processed pickle: {missing}")

    n_observed = len(data["coord_cols_observed"])
    n_active = len(data["coord_cols_active"])
    slices = data["active_modality_slices"]

    if expected_observed_total is not None and n_observed != expected_observed_total:
        print(f"⚠️  Observed features = {n_observed}, expected {expected_observed_total}")

    if expected_active_total is not None and n_active != expected_active_total:
        print(f"⚠️  Active features = {n_active}, expected {expected_active_total}")

    raw_to_active = data["raw_to_active_indices"]
    if len(raw_to_active) != n_active:
        raise AssertionError("raw_to_active_indices length does not match coord_cols_active length.")

    if slices["body"][1] != n_active:
        raise AssertionError("active_modality_slices do not end at n_active.")

    video_ids = data["video_ids"]
    video_labels = data["video_labels"]
    video_to_frames = data["video_to_frames"]

    if len(video_ids) != len(video_labels):
        raise AssertionError("video_ids and video_labels length mismatch.")

    if len(video_ids) != len(video_to_frames):
        raise AssertionError("video_ids and video_to_frames length mismatch.")

    # Lightweight data integrity check on first few videos
    sample_vids = list(video_ids[: min(5, len(video_ids))])
    for vid in sample_vids:
        arr = video_to_frames[str(vid)]["features"]
        if arr.shape[1] != n_observed:
            raise AssertionError(
                f"Video {vid} feature dimension = {arr.shape[1]}, expected observed={n_observed}"
            )
        if not np.isfinite(arr).all():
            raise AssertionError(f"Non-finite values found in sample video {vid}.")

    print(f"\n{'=' * 70}")
    print(f"SANITY CHECK PASSED: {dataset_name}")
    print(f"{'=' * 70}")
    print(f"  Videos:             {len(video_ids)}")
    print(f"  Observed features:  {n_observed}")
    print(f"  Active features:    {n_active}")
    print(f"  Active slices:      {slices}")
    print(f"  Preprocessing ver.: {data['metadata'].get('preprocessing_version')}")
    print(f"{'=' * 70}\n")

    return {
        "dataset_name": dataset_name,
        "videos": int(len(video_ids)),
        "observed_features": int(n_observed),
        "active_features": int(n_active),
        "active_modality_slices": slices,
        "sanity_check": True,
    }


# ============================================================
# CLI / Notebook Runner
# ============================================================
if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("MSTGNet DATA PREPROCESSING / PATCH PIPELINE")
    print(f"Version: {PREPROCESSING_VERSION}")
    print("=" * 70)

    dataset_paths = {
        "I3D": "data/raw/LandMarkData/LandmarkDataset_I3D.csv",
        "RLT": "data/raw/LandMarkData/LandmarkDataset_RLT.csv",
    }

    IRIS_POLICY = "standard_only"

    # ------------------------------------------------------------
    # IMPORTANT:
    # For revision v3, we usually only need to patch existing large
    # processed pickles. Set this to False only if raw preprocessing
    # must be rerun from CSV.
    # ------------------------------------------------------------
    PATCH_EXISTING_PICKLES_ONLY = True

    results = {}

    if PATCH_EXISTING_PICKLES_ONLY:
        print("\nMode: PATCH_EXISTING_PICKLES_ONLY = True")
        print("Existing processed .pkl files will be backed up and patched.")
        print("Raw CSV preprocessing will NOT be rerun.\n")

        for dataset_name in dataset_paths.keys():
            try:
                patch_result = patch_existing_processed_pickle(
                    dataset_name,
                    processed_dir=str(PROCESSED_DIR),
                    backup=True,
                    backup_suffix="_backup_v1",
                    expected_active_total=1503,
                )
                sanity_result = sanity_check_processed_pickle(
                    dataset_name,
                    processed_dir=str(PROCESSED_DIR),
                    expected_observed_total=1533,
                    expected_active_total=1503,
                )
                results[dataset_name] = {
                    "patch": patch_result,
                    "sanity": sanity_result,
                }
            except Exception as e:
                print(f"❌ Error patching {dataset_name}: {str(e)}")
                import traceback
                traceback.print_exc()

    else:
        print("\nMode: FULL RAW PREPROCESSING")
        print("Raw CSV files will be loaded and processed from scratch.\n")

        for dataset_name, file_path in dataset_paths.items():
            try:
                result = preprocess_for_mstgnet(
                    file_path,
                    dataset_name,
                    iris_policy=IRIS_POLICY,
                    require_integer_frame=False,
                    allow_duplicate_frames=True,
                    deduplicate_frames=False,
                    dedup_keep="first",
                    sort_video_ids=False,
                )
                if result is not None:
                    results[dataset_name] = result
                    print(f"✅ {dataset_name} preprocessing completed!\n")
                else:
                    print(f"❌ {dataset_name} preprocessing failed!\n")
            except Exception as e:
                print(f"❌ Error processing {dataset_name}: {str(e)}")
                import traceback
                traceback.print_exc()

    print("\n" + "=" * 70)
    print("ALL PROCESSING COMPLETED")
    print("=" * 70)
    print(f"Successfully processed/patched: {len(results)}/{len(dataset_paths)} datasets\n")

    for dataset_name in dataset_paths.keys():
        try:
            check = sanity_check_processed_pickle(
                dataset_name,
                processed_dir=str(PROCESSED_DIR),
                expected_observed_total=1533,
                expected_active_total=1503,
            )
            print(f"✅ {dataset_name}:")
            print(f"   Videos: {check['videos']}")
            print(f"   Observed features: {check['observed_features']}")
            print(f"   Active features:   {check['active_features']}")
            print(f"   Active slices:     {check['active_modality_slices']}")
            print()
        except Exception as e:
            print(f"❌ Final sanity check failed for {dataset_name}: {e}")

    print("📁 Output files:")
    print("   - data/processed/mstgnet_I3D.pkl")
    print("   - data/processed/mstgnet_RLT.pkl")
    print("   - data/processed/mstgnet_I3D_backup_v1.pkl")
    print("   - data/processed/mstgnet_RLT_backup_v1.pkl")
    print("   - data/processed/mstgnet_*_summary.txt if full preprocessing is rerun")
    print("\n✅ Data ready for 02_MSTGNet_experiment.ipynb")
    print("=" * 70)


MSTGNet DATA PREPROCESSING / PATCH PIPELINE
Version: 10.2_revision_ready_integrity_first

Mode: PATCH_EXISTING_PICKLES_ONLY = True
Existing processed .pkl files will be backed up and patched.
Raw CSV preprocessing will NOT be rerun.

✓ Backup created: data\processed\mstgnet_I3D_backup_v1.pkl

PATCHING EXISTING PROCESSED PICKLE: I3D
Loading: data\processed\mstgnet_I3D.pkl
✓ Derived active feature metadata
  coord_cols_observed: 1533
  coord_cols_active:   1503
  active slices:       {'face': [0, 1374], 'eyes': [1374, 1404], 'body': [1404, 1503]}
✓ Patched and saved: data\processed\mstgnet_I3D.pkl
  File size: 3789.13 MB
Patch summary:
  Observed features: 1533
  Active features:   1503
  Active slices:     {'face': [0, 1374], 'eyes': [1374, 1404], 'body': [1404, 1503]}


SANITY CHECK PASSED: I3D
  Videos:             1568
  Observed features:  1533
  Active features:    1503
  Active slices:      {'face': [0, 1374], 'eyes': [1374, 1404], 'body': [1404, 1503]}
  Preprocessing ver.: 10.2